# 🦺 SafeStride AI — Local VS Code Inference
> **YOLOv8m** · General object awareness scaled for local Windows execution.

**General classes (pretrained COCO):** `person`, `bicycle`, `car`, `motorcycle`, `bus`, `truck`, `traffic light`, `stop sign`, `cat`, `dog`

### VS Code Requirements:
1. Select your Python kernel in the top-right corner.
2. Run the dependency cell below if you haven't installed requirements yet.
3. Ensure your webcam is available.

In [ ]:
# 1. Install Dependencies locally (Run this once)
# %pip install ultralytics opencv-python pyttsx3 ipykernel ipywidgets pywin32

In [ ]:
import cv2
import numpy as np
import time
import os
import json
import multiprocessing  # <--- Switching to Multiprocessing for stability
import pythoncom
import ipywidgets as widgets
from collections import defaultdict, Counter
from pathlib import Path

import pyttsx3
from ultralytics import YOLO

print('All imports successful ✓')

In [ ]:
# ─── Paths ────────────────────────────────────────────────────────────────────
MODEL_PATH         = 'yolov8m.pt'      
INPUT_SOURCE       = 0                 
LOOP_VIDEO         = True
AUDIO_DIR          = './safestride_audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

# ─── COCO class IDs ───────────────────────────────────────────────────────────
RELEVANT_COCO_CLASSES = {
    0:  'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus',
    7:  'truck', 9: 'traffic light', 11: 'stop sign', 15: 'cat', 16: 'dog',
}

# ─── Detection & Alert Settings ───────────────────────────────────────────────
CONF_THRESHOLD         = 0.50
IOU_THRESHOLD          = 0.45
COOLDOWN_SEC           = 1.0   # Set to 1.0 for much more continuous feedback

# ─── TTS Settings ─────────────────────────────────────────────────────────────
TTS_RATE   = 185
TTS_VOLUME = 1.0

print(f'Configuration loaded. Using model: {MODEL_PATH}')

In [ ]:
def get_direction(cx, frame_w):
    ratio = cx / frame_w
    if ratio < 0.35: return 'to your left'
    if ratio > 0.65: return 'to your right'
    return 'ahead'

def get_proximity(bbox_area, frame_area):
    ratio = bbox_area / frame_area
    if ratio > 0.12: return 'very close'
    if ratio > 0.04: return 'close'
    return ''

def build_alert_text(class_name, cx, frame_w, bbox_area, frame_area):
    direction = get_direction(cx, frame_w)
    proximity = get_proximity(bbox_area, frame_area)
    text = f"{class_name.capitalize()} {proximity} {direction}"
    return ' '.join(text.split())

print('Direction & Proximity logic defined ✓')

In [ ]:
class AlertManager:
    def __init__(self, cooldown):
        self.cooldown = cooldown
        self._last = defaultdict(float)
    
    def should_alert(self, class_name):
        now = time.time()
        if now - self._last[class_name] >= self.cooldown:
            self._last[class_name] = now
            return True
        return False

In [ ]:
def tts_worker(q, rate, volume):
    """This function runs in a completely separate process to avoid Jupyter hangs."""
    pythoncom.CoInitialize()
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', rate)
        engine.setProperty('volume', volume)
        while True:
            text = q.get()
            if text is None: break
            engine.say(text)
            engine.runAndWait()
    except Exception as e:
        print(f"[TTS Process Error] {e}")
    finally:
        pythoncom.CoUninitialize()

class TTSEngine:
    def __init__(self, rate=175, volume=1.0):
        self._q = multiprocessing.Queue()
        self._process = multiprocessing.Process(
            target=tts_worker, 
            args=(self._q, rate, volume), 
            daemon=True
        )
        self._process.start()
        print("[TTS] Voice engine process started")

    def speak(self, text):
        # Clear queue if it gets too backed up (keeps alerts current)
        while self._q.qsize() > 3:
            try: self._q.get_nowait()
            except: break
        self._q.put(text)

    def stop(self):
        self._q.put(None)
        self._process.join(timeout=2)
        if self._process.is_alive():
            self._process.terminate()

print('Local TTSEngine (Multiprocessing) ready ✓')

In [ ]:
def run_inference():
    model = YOLO(MODEL_PATH)
    tts = TTSEngine(rate=TTS_RATE)
    alert_mgr = AlertManager(COOLDOWN_SEC)
    
    cap = cv2.VideoCapture(INPUT_SOURCE)
    if not cap.isOpened():
        print(f"Error: Could not open video source '{INPUT_SOURCE}'.")
        return

    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_area = frame_w * frame_h

    print(f"Starting inference on {INPUT_SOURCE}... Press 'q' to stop.")
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                if LOOP_VIDEO and isinstance(INPUT_SOURCE, str):
                    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                    continue
                else:
                    break

            results = model(frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, 
                            classes=list(RELEVANT_COCO_CLASSES.keys()), verbose=False)[0]
            
            for box in results.boxes:
                cls_id = int(box.cls[0])
                cname = RELEVANT_COCO_CLASSES.get(cls_id, 'object')
                conf_val = float(box.conf[0])
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx = (x1 + x2) / 2
                bbox_area = (x2 - x1) * (y2 - y1)

                if alert_mgr.should_alert(cname):
                    alert_text = build_alert_text(cname, cx, frame_w, bbox_area, frame_area)
                    tts.speak(alert_text)
                    print(f"[Alert] {alert_text} ({conf_val:.2f})")

            annotated_frame = results.plot()
            cv2.imshow("SafeStride Local Inference", annotated_frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        tts.stop()
        print("Inference stopped.")

print('Inference function defined ✓')

In [ ]:
# ▶ RUN THIS CELL TO START
run_inference()